# quant-lab research template

The canonical end-to-end research flow. Copy this notebook for new experiments.

```
load BTC data -> inspect -> validate -> features -> signals
    -> VectorBT backtest -> metrics -> plots -> parameter sweep -> compare
```

This is a local research environment. It does not place live orders.

In [ ]:
import polars as pl

from quant_lab.config import get_config
from quant_lab.data.schemas import DatasetType
from quant_lab.data.storage import DatasetKey, read_dataset
from quant_lab.data.validate import validate
from quant_lab.features import FeatureSpec, compute_features
from quant_lab.strategies import build_strategy
from quant_lab.backtest.engine import run_backtest
from quant_lab.optimization import run_sweep, parameter_grid, best_by
from quant_lab.analytics import bootstrap_returns, monte_carlo_trades

cfg = get_config()
SYMBOL, TIMEFRAME = "BTCUSDT", "1m"
START, END = "2024-01-01", "2024-04-01"
cfg.paths.data_normalized

## 1. Load BTC data

If this is empty, run:

```bash
quantlab data build binance --market futures --symbol BTCUSDT \
    --interval 1m --start 2024-01-01 --end 2024-04-01
```

In [ ]:
key = DatasetKey("binance", "futures", DatasetType.BARS, SYMBOL, TIMEFRAME)
df = read_dataset(cfg.paths.data_normalized, key, start=START, end=END)
print(f"{df.height:,} bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head(3)

## 2. Inspect and validate data quality

Validation reports; it never silently repairs.

In [ ]:
report = validate(df, key, timeframe=TIMEFRAME)
print(report.format())

In [ ]:
df.select(
    pl.col("close").min().alias("min_close"),
    pl.col("close").max().alias("max_close"),
    pl.col("volume").mean().alias("mean_volume"),
    pl.col("timestamp").diff().dt.total_seconds().median().alias("median_step_s"),
)

## 3. Calculate features

Features are causal by construction (trailing windows only).

In [ ]:
features = compute_features(df, [
    FeatureSpec("ema", {"span": 100}),
    FeatureSpec("atr", {"period": 14}),
    FeatureSpec("rsi", {"period": 14}),
    FeatureSpec("realized_volatility", {"window": 60}),
    FeatureSpec("relative_volume", {"window": 20}),
])
features.select("timestamp", "close", "ema", "atr", "rsi", "relative_volume").tail(3)

## 4. Generate signals

The reference strategy: `close > EMA(100)` and a volatility breakout and a volume expansion.

In [ ]:
strategy = build_strategy("volatility_breakout", ema_span=100, atr_period=14, volume_multiplier=1.5)
signals = strategy.prepare(df)
signals.select("timestamp", "close", "entries", "exits").filter(pl.col("entries")).head(5)

## 5. Run the VectorBT backtest

Timing contract: signal at bar `t` close, fill at bar `t+1` open, minus fees and slippage.

In [ ]:
result = run_backtest(
    df, strategy, symbol=SYMBOL, timeframe=TIMEFRAME, dataset_id=key.dataset_id,
    start=START, end=END,
)
from quant_lab.reporting import format_summary
print(format_summary(result))

## 6. Metrics

Every metric is exposed; there is no opaque strategy score.

In [ ]:
pl.DataFrame([{"metric": k, "value": v} for k, v in result.metrics.items()])

In [ ]:
result.trades.head(10)

## 7. Equity and drawdown

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(result.equity.index, result.equity.values)
ax[0].set_title(f"Equity — {SYMBOL} {TIMEFRAME}")
ax[1].fill_between(result.drawdown.index, result.drawdown.values * 100, 0, color="red", alpha=0.5)
ax[1].set_title("Drawdown (%)")
plt.tight_layout()

## 8. Parameter sweep

Use the strategy's declared grid, or override it.

In [ ]:
grid = parameter_grid(type(strategy), overrides={"ema_span": [50, 100, 150], "volume_multiplier": [1.0, 1.5]})
sweep = run_sweep(df, type(strategy), grid, symbol=SYMBOL, timeframe=TIMEFRAME, dataset_id=key.dataset_id)
sweep.sort("sharpe", descending=True).head(10)

## 9. Compare results

In [ ]:
print("best by Sharpe:", best_by(sweep, "sharpe"))
sweep.select("ema_span", "volume_multiplier", "total_return", "sharpe", "max_drawdown", "trade_count")

## 10. Robustness

Distributions are exposed directly rather than collapsed into a score.

In [ ]:
boot = bootstrap_returns(result.returns, n=500, block=10, periods_per_year=result.periods_per_year)
print(boot.format())
mc = monte_carlo_trades(result.trades["pnl"], n=500)
print(mc.format())